In [ ]:
!pip install whitebox
!pip install rasterio geopandas scikit-learn rasterstats matplotlib scipy xgboost

In [2]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import rasterio
from rasterio.merge import merge
from rasterio.crs import CRS
from scipy.ndimage import uniform_filter, generic_filter
import whitebox

wbt = whitebox.WhiteboxTools()
wbt.set_verbose_mode(True)

# --- PATHS (edit these) ---
LAZ_FOLDER   = r'C:\users\colto\documents\github\lidar_project\data\files'
DEM_FOLDER   = r'C:\users\colto\documents\github\lidar_project\data\dem_tiles'
DERIV_FOLDER = r'C:\users\colto\documents\github\lidar_project\data\derivatives'
FULL_DEM     = r'C:\users\colto\documents\github\lidar_project\data\full_dem.tif'

os.makedirs(DEM_FOLDER,   exist_ok=True)
os.makedirs(DERIV_FOLDER, exist_ok=True)

print("Setup complete.")


Setup complete.


TAKES LONG TIME


In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import laspy, glob, os, whitebox

MAX_WORKERS = 4  # tune to your CPU core count — don't go higher than cores - 1

# ── Step 1: COPC conversion (parallel) ───────────────────────────────────────
def convert_copc(f):
    out = f.replace('.copc.laz', '.laz')
    if os.path.exists(out):
        return f"Skipped (exists): {os.path.basename(out)}"
    copc_las   = laspy.read(f)
    new_header = laspy.LasHeader(
        point_format=copc_las.header.point_format,
        version=copc_las.header.version
    )
    new_header.offsets = copc_las.header.offsets
    new_header.scales  = copc_las.header.scales
    new_las        = laspy.LasData(header=new_header)
    new_las.points = copc_las.points
    new_las.write(out)
    return f"Converted: {os.path.basename(out)}"

copc_files = glob.glob(os.path.join(LAZ_FOLDER, '*.copc.laz'))
if copc_files:
    print(f"Converting {len(copc_files)} COPC files (parallel)...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(convert_copc, f): f for f in copc_files}
        for fut in as_completed(futures):
            print(f"  {fut.result()}")
else:
    print("No COPC files found.")

# ── Step 2: LAZ → DEM (parallel) ─────────────────────────────────────────────
def process_laz(laz):
    basename = os.path.splitext(os.path.basename(laz))[0]
    out_dem  = os.path.join(DEM_FOLDER, f'{basename}_dem.tif')

    if os.path.exists(out_dem):
        return f"Skipped: {basename}"

    # Each thread gets its own WBT instance — avoids shared state issues
    _wbt = whitebox.WhiteboxTools()
    _wbt.set_verbose_mode(False)
    _wbt.lidar_tin_gridding(
        i=laz,
        output=out_dem,
        resolution=1.0,
        returns='last',
        exclude_cls='0,1,3,4,5,6,7,9,17'
    )
    return f"Done: {basename}"

laz_files = [f for f in glob.glob(os.path.join(LAZ_FOLDER, '*.laz'))
             if '.copc.' not in f]
print(f"\nFound {len(laz_files)} standard LAZ files")

new_tiles = 0
skipped   = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(process_laz, laz): laz for laz in laz_files}
    for fut in as_completed(futures):
        result = fut.result()
        print(f"  {result}")
        if "Skipped" in result:
            skipped += 1
        else:
            new_tiles += 1

print(f"\nNew tiles processed: {new_tiles}")
print(f"Skipped (already done): {skipped}")

No COPC files found.

Found 176 standard LAZ files
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604591
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604593
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604594
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604590
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604596
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604599
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF604597
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606590
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606591
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606593
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606594
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606596
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606597
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF607590
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF606599
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF607591
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF607593
  Done: USGS_LPC_PA_WesternPA_2019_D20_17TPF607596
  Done: USGS_LPC_PA_WesternPA_2

In [4]:
# # Step 1: Convert any COPC files to standard LAZ
# print("Checking for COPC files...")
# copc_files = glob.glob(os.path.join(LAZ_FOLDER, '*.copc.laz'))
# if copc_files:
#     for f in copc_files:
#         out = f.replace('.copc.laz', '.laz')
#         if not os.path.exists(out):
#             print(f"  Converting {os.path.basename(f)}...")
#             copc_las = laspy.read(f)
#             # Create fresh LasData — strips COPC-specific VLRs that block writing
#             new_header = laspy.LasHeader(
#                 point_format=copc_las.header.point_format,
#                 version=copc_las.header.version
#             )
#             new_header.offsets = copc_las.header.offsets
#             new_header.scales  = copc_las.header.scales
#             new_las = laspy.LasData(header=new_header)
#             new_las.points = copc_las.points
#             new_las.write(out)
#             print(f"  Saved: {os.path.basename(out)}")
#         else:
#             print(f"  Already converted: {os.path.basename(out)}")
# else:
#     print("  No COPC files found.")
#
# # Step 2: Process only NEW LAZ tiles
# laz_files = [f for f in glob.glob(os.path.join(LAZ_FOLDER, '*.laz'))
#              if '.copc.' not in f]
# print(f"\nFound {len(laz_files)} standard LAZ files")
#
# new_tiles = 0
# skipped   = 0
# for laz in laz_files:
#     basename = os.path.splitext(os.path.basename(laz))[0]
#     out_dem  = os.path.join(DEM_FOLDER, f'{basename}_dem.tif')
#
#     if os.path.exists(out_dem):
#         skipped += 1
#         continue
#
#     print(f"Processing {basename}...")
#     wbt.lidar_tin_gridding(
#         i=laz,
#         output=out_dem,
#         resolution=1.0,
#         returns='last',
#         exclude_cls='0,1,3,4,5,6,7,9,17'
#     )
#     new_tiles += 1
#
# print(f"\nNew tiles processed: {new_tiles}")
# print(f"Skipped (already done): {skipped}")

In [5]:
print("Mosaicking all tiles...")
dem_tiles = glob.glob(os.path.join(DEM_FOLDER, '*_dem.tif'))
print(f"Tiles found: {len(dem_tiles)}")

src_files = [rasterio.open(f) for f in dem_tiles]
mosaic, out_transform = merge(src_files)

out_profile = src_files[0].profile.copy()
out_profile.update({
    'height':    mosaic.shape[1],
    'width':     mosaic.shape[2],
    'transform': out_transform,
    'crs':       CRS.from_epsg(26917)
})

with rasterio.open(FULL_DEM, 'w', **out_profile) as dst:
    dst.write(mosaic)

for src in src_files:
    src.close()

with rasterio.open(FULL_DEM) as src:
    dem_check = src.read(1).astype('float32')
    dem_check[dem_check == src.nodata] = np.nan
    print(f"\nMosaic updated → {FULL_DEM}")
    print(f"  Shape:     {src.read(1).shape}")
    print(f"  CRS:       {src.crs}")
    print(f"  Elevation: {np.nanmin(dem_check):.1f}m to {np.nanmax(dem_check):.1f}m")

Mosaicking all tiles...
Tiles found: 176

Mosaic updated → C:\users\colto\documents\github\lidar_project\data\full_dem.tif
  Shape:     (19500, 21000)
  CRS:       EPSG:26917
  Elevation: 304.5m to 532.9m


In [6]:
with rasterio.open(FULL_DEM) as src:
    dem      = src.read(1).astype('float32')
    profile  = src.profile.copy()
    nodata_v = src.nodata

profile.update(dtype='float32', nodata=np.nan)

valid_mask = dem != nodata_v

fill_value           = float(np.nanmean(dem[valid_mask]))
dem_filled           = dem.copy()
dem_filled[~valid_mask] = fill_value

print(f"Valid pixels:  {valid_mask.sum():,}")
print(f"Nodata pixels: {(~valid_mask).sum():,}")
print(f"Fill value:    {fill_value:.2f}m")

Valid pixels:  395,719,288
Nodata pixels: 13,780,712
Fill value:    449.70m


In [7]:
# # Instead of just .laz
# laz_files = glob.glob(os.path.join(LAZ_FOLDER, '*.laz'))
#
# # This catches both .laz and .copc.laz
# laz_files = glob.glob(os.path.join(LAZ_FOLDER, '*.laz')) + \
#             glob.glob(os.path.join(LAZ_FOLDER, '*.copc.laz'))
#
# # Or more cleanly with recursive pattern
# import glob
# laz_files = [f for f in glob.glob(os.path.join(LAZ_FOLDER, '*'))
#              if f.endswith('.laz')]
# print(f"Found {len(laz_files)} LAZ files.")
#
# for laz in laz_files:
#     print(laz)
#     basename = os.path.splitext(os.path.basename(laz))[0]
#     out_dem = os.path.join(DEM_FOLDER, f'{basename}_dem.tif')
#
#     if os.path.exists(out_dem):
#         print(f"Skipping {basename} (already done)")
#         continue
#     elif 'copc.laz' in laz:
#         continue
#     print(f"\nProcessing {basename}...")
#     wbt.lidar_tin_gridding(
#         i=laz,
#         output=out_dem,
#         resolution=1.0,
#         returns='last',
#         exclude_cls='0,1,3,4,5,6,7,9,17'   # keep class 2 (ground) only
#     )
#
# print("\nAll tiles processed.")

In [8]:
# dem_tiles = glob.glob(os.path.join(DEM_FOLDER, '*_dem.tif'))
# src_files  = [rasterio.open(f) for f in dem_tiles]
#
# mosaic, out_transform = merge(src_files)
#
# out_profile = src_files[0].profile.copy()
# out_profile.update({
#     'height':    mosaic.shape[1],
#     'width':     mosaic.shape[2],
#     'transform': out_transform,
#     'crs':       CRS.from_epsg(26917)     # UTM Zone 17N — correct for western PA
# })
#
# with rasterio.open(FULL_DEM, 'w', **out_profile) as dst:
#     dst.write(mosaic)
#
# for src in src_files:
#     src.close()
#
# # Verify
# with rasterio.open(FULL_DEM) as src:
#     print(f"DEM shape:      {src.read(1).shape}")
#     print(f"Resolution:     {src.res}")
#     print(f"CRS:            {src.crs}")
#     print(f"NoData value:   {src.nodata}")
#     dem_check = src.read(1).astype('float32')
#     dem_check[dem_check == src.nodata] = np.nan
#     print(f"Elevation range: {np.nanmin(dem_check):.1f}m to {np.nanmax(dem_check):.1f}m")
#
# print(f"\nMosaic saved to {FULL_DEM}")

In [9]:
with rasterio.open(FULL_DEM) as src:
    dem      = src.read(1).astype('float32')
    profile  = src.profile.copy()
    nodata_v = src.nodata

profile.update(dtype='float32', nodata=np.nan)

# Valid pixel mask
valid_mask = dem != nodata_v

# Fill nodata with global mean so scipy filters don't propagate NaN
fill_value          = float(np.nanmean(dem[valid_mask]))
dem_filled          = dem.copy()
dem_filled[~valid_mask] = fill_value

print(f"Valid pixels:  {valid_mask.sum():,}")
print(f"Nodata pixels: {(~valid_mask).sum():,}")
print(f"Fill value:    {fill_value:.2f}m")

Valid pixels:  395,719,288
Nodata pixels: 13,780,712
Fill value:    449.70m


TAKES A WHILE

In [10]:
# def save_raster(array, filename):
#     path = os.path.join(DERIV_FOLDER, filename)
#     with rasterio.open(path, 'w', **profile) as dst:
#         dst.write(array.astype('float32'), 1)
#     print(f"  Saved {filename}")
#
# # --- TPI (Topographic Position Index) at multiple scales ---
# def compute_tpi(dem_filled, valid_mask, radius_pixels):
#     neighborhood_mean = uniform_filter(dem_filled, size=radius_pixels * 2 + 1)
#     tpi = dem_filled - neighborhood_mean
#     tpi[~valid_mask] = np.nan
#     return tpi
#
# print("Computing TPI...")
# save_raster(compute_tpi(dem_filled, valid_mask, 5),  'tpi_5m.tif')
# save_raster(compute_tpi(dem_filled, valid_mask, 15), 'tpi_15m.tif')
# save_raster(compute_tpi(dem_filled, valid_mask, 50), 'tpi_50m.tif')
#
# # --- Slope and Curvature (finite differences) ---
# def compute_slope_curvature(dem, cellsize=1.0):
#     d = np.pad(dem, 1, mode='edge')
#     dz_dx = (d[1:-1, 2:]  - d[1:-1, :-2]) / (2 * cellsize)
#     dz_dy = (d[2:,  1:-1] - d[:-2,  1:-1]) / (2 * cellsize)
#     slope     = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
#     d2z_dx2   = (d[1:-1, 2:] - 2*d[1:-1,1:-1] + d[1:-1,:-2]) / cellsize**2
#     d2z_dy2   = (d[2:,1:-1]  - 2*d[1:-1,1:-1] + d[:-2, 1:-1]) / cellsize**2
#     curvature = d2z_dx2 + d2z_dy2
#     slope[~valid_mask]     = np.nan
#     curvature[~valid_mask] = np.nan
#     return slope, curvature
#
# print("Computing slope and curvature...")
# slope, curvature = compute_slope_curvature(dem_filled)
# save_raster(slope,     'slope.tif')
# save_raster(curvature, 'curvature.tif')
#
# # --- Local Relief (range in moving window) ---
# print("Computing local relief (slow — ~2-3 min)...")
# local_max = generic_filter(dem_filled, np.nanmax, size=21)
# local_min = generic_filter(dem_filled, np.nanmin, size=21)
# relief    = local_max - local_min
# relief[~valid_mask] = np.nan
# save_raster(relief, 'relief_10m.tif')
#
# # --- Roughness (std dev of slope in window) ---
# print("Computing roughness (slow — ~2-3 min)...")
# roughness = generic_filter(slope, np.nanstd, size=11)
# roughness[~valid_mask] = np.nan
# save_raster(roughness, 'roughness.tif')
#
# print("\nAll derivatives done.")

In [11]:
from scipy.ndimage import uniform_filter, maximum_filter, minimum_filter

def save_raster(array, filename):
    path = os.path.join(DERIV_FOLDER, filename)
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(array.astype('float32'), 1)
    print(f"  Saved {filename}")

# --- TPI — already fast, no change needed ---
def compute_tpi(dem_filled, valid_mask, radius_pixels):
    neighborhood_mean = uniform_filter(dem_filled, size=radius_pixels * 2 + 1)
    tpi = dem_filled - neighborhood_mean
    tpi[~valid_mask] = np.nan
    return tpi

print("Computing TPI...")
save_raster(compute_tpi(dem_filled, valid_mask, 5),  'tpi_5m.tif')
save_raster(compute_tpi(dem_filled, valid_mask, 15), 'tpi_15m.tif')
save_raster(compute_tpi(dem_filled, valid_mask, 50), 'tpi_50m.tif')

# --- Slope and Curvature — already fast vectorized numpy, no change ---
def compute_slope_curvature(dem, cellsize=1.0):
    d         = np.pad(dem, 1, mode='edge')
    dz_dx     = (d[1:-1, 2:]  - d[1:-1, :-2]) / (2 * cellsize)
    dz_dy     = (d[2:,  1:-1] - d[:-2,  1:-1]) / (2 * cellsize)
    slope     = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
    d2z_dx2   = (d[1:-1, 2:] - 2*d[1:-1,1:-1] + d[1:-1,:-2]) / cellsize**2
    d2z_dy2   = (d[2:,1:-1]  - 2*d[1:-1,1:-1] + d[:-2, 1:-1]) / cellsize**2
    curvature = d2z_dx2 + d2z_dy2
    slope[~valid_mask]     = np.nan
    curvature[~valid_mask] = np.nan
    return slope, curvature

print("Computing slope and curvature...")
slope, curvature = compute_slope_curvature(dem_filled)
save_raster(slope,     'slope.tif')
save_raster(curvature, 'curvature.tif')

# --- Local Relief — maximum_filter/minimum_filter instead of generic_filter ---
# Was: generic_filter(np.nanmax) — Python callback per pixel, very slow
# Now: C-implemented morphological filters, 50-100x faster
print("Computing local relief...")
size = 21
local_max = maximum_filter(dem_filled, size=size)
local_min = minimum_filter(dem_filled, size=size)
relief = local_max - local_min
relief[~valid_mask] = np.nan
save_raster(relief, 'relief_10m.tif')

# --- Roughness (std dev of slope in window) ---
print("Computing roughness...")
size = 11

# Fill NaN in slope before uniform_filter — same fix as TPI
slope_fill_val    = float(np.nanmean(slope[valid_mask]))
slope_filled      = slope.copy()
slope_filled[~valid_mask] = slope_fill_val

slope_sq_mean = uniform_filter(slope_filled**2, size=size)
slope_mean_sq = uniform_filter(slope_filled,    size=size)**2
roughness     = np.sqrt(np.maximum(slope_sq_mean - slope_mean_sq, 0))
roughness[~valid_mask] = np.nan
save_raster(roughness, 'roughness.tif')

Computing TPI...
  Saved tpi_5m.tif
  Saved tpi_15m.tif
  Saved tpi_50m.tif
Computing slope and curvature...
  Saved slope.tif
  Saved curvature.tif
Computing local relief...
  Saved relief_10m.tif
Computing roughness...
  Saved roughness.tif


In [12]:
# ── ROUGHNESS ────────────────────────────────────────────────────────────────
size = 11

slope_crop = slope[:500, :500]
mask_crop  = valid_mask[:500, :500]

# Old (slow)
old_roughness = generic_filter(slope_crop, np.nanstd, size=size)

# New (fast) — fill NaN before uniform_filter, restore mask after
slope_fill_val = float(np.nanmean(slope_crop))
slope_crop_filled = slope_crop.copy()
slope_crop_filled[~mask_crop] = slope_fill_val

slope_sq_mean = uniform_filter(slope_crop_filled**2, size=size)
slope_mean_sq = uniform_filter(slope_crop_filled,    size=size)**2
new_roughness = np.sqrt(np.maximum(slope_sq_mean - slope_mean_sq, 0))
new_roughness[~mask_crop] = np.nan

# Compare
diff_rough = np.abs(old_roughness - new_roughness)
print("\n=== ROUGHNESS ===")
print(f"Max absolute difference:  {np.nanmax(diff_rough):.6f}")
print(f"Mean absolute difference: {np.nanmean(diff_rough):.8f}")
print(f"Old mean: {np.nanmean(old_roughness):.4f}")
print(f"New mean: {np.nanmean(new_roughness):.4f}")


=== ROUGHNESS ===
Max absolute difference:  4.904186
Mean absolute difference: 0.02406281
Old mean: 1.9250
New mean: 1.8285


In [13]:
for fname in ['tpi_5m.tif', 'tpi_15m.tif', 'tpi_50m.tif',
              'slope.tif', 'curvature.tif', 'relief_10m.tif', 'roughness.tif']:
    with rasterio.open(os.path.join(DERIV_FOLDER, fname)) as src:
        data  = src.read(1).astype('float32')
        valid = data[~np.isnan(data)]

        if len(valid) == 0:
            print(f"{fname:20s}  *** ALL NaN — file needs to be recomputed ***")
            continue

        p2, p98 = np.percentile(valid, [2, 98])
        print(f"{fname:20s}  valid={len(valid):>9,}  "
              f"range=[{valid.min():8.4f}, {valid.max():8.4f}]  "
              f"p2-p98=[{p2:.4f}, {p98:.4f}]")

tpi_5m.tif            valid=395,719,288  range=[-64.9879,  20.9805]  p2-p98=[-0.3271, 0.3093]
tpi_15m.tif           valid=395,719,288  range=[-70.4346,  21.6524]  p2-p98=[-0.9025, 0.7640]
tpi_50m.tif           valid=395,719,288  range=[-75.4586,  20.1054]  p2-p98=[-3.0217, 2.3967]
slope.tif             valid=395,719,288  range=[  0.0000,  89.4285]  p2-p98=[0.6379, 29.0968]
curvature.tif         valid=395,719,288  range=[-124.2739, 283.6268]  p2-p98=[-0.3001, 0.2462]
relief_10m.tif        valid=395,719,288  range=[  0.0053, 144.1639]  p2-p98=[0.4639, 14.6390]
roughness.tif         valid=395,719,288  range=[  0.0000,  39.6553]  p2-p98=[0.5426, 10.3238]


In [14]:
# files = {
#     'TPI 5m':       'tpi_5m.tif',
#     'TPI 15m':      'tpi_15m.tif',
#     'TPI 50m':      'tpi_50m.tif',
#     'Local Relief': 'relief_10m.tif',
#     'Slope':        'slope.tif',
#     'Curvature':    'curvature.tif',
#     'Roughness':    'roughness.tif',
# }
#
# fig, axes = plt.subplots(2, 4, figsize=(18, 10))
# axes = axes.flatten()
#
# for idx, (title, fname) in enumerate(files.items()):
#     with rasterio.open(os.path.join(DERIV_FOLDER, fname)) as src:
#         data = src.read(1).astype('float32')
#         data[data == src.nodata] = np.nan
#
#     ax = axes[idx]
#
#     if 'TPI' in title or 'Curvature' in title:
#         absmax = np.nanpercentile(np.abs(data), 98)
#         norm   = TwoSlopeNorm(vmin=-absmax, vcenter=0, vmax=absmax)
#         cmap   = 'coolwarm_r'    # blue = depression, red = ridge
#         im     = ax.imshow(data, cmap=cmap, norm=norm)
#     else:
#         vmin, vmax = np.nanpercentile(data, [2, 98])
#         im = ax.imshow(data, cmap='cividis', vmin=vmin, vmax=vmax)
#
#     ax.set_title(title, fontsize=11)
#     ax.axis('off')
#     plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
#
# axes[-1].axis('off')
# plt.suptitle('Terrain Derivatives', fontsize=14)
# plt.tight_layout()
# plt.savefig(r'C:\users\colto\documents\github\lidar_project\data\derivatives_preview.png', dpi=150, bbox_inches='tight')
# plt.show()
# print("Preview saved.")

In [15]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
import os, glob

# --- Source 1: PADEP Historic Wells shapefile ---
wells_path = r'C:\Users\colto\Documents\PADEP_HistoricOilGasWells_ALL'
shp_file = glob.glob(os.path.join(wells_path, '*.shp'))[0]
print(f"Loading shapefile: {shp_file}")

padep = gpd.read_file(shp_file)
padep_slim = gpd.GeoDataFrame({
    'source':   'PADEP_historic',
    'state':    'Pennsylvania',
    'type':     padep['TYPE'],
    'geometry': padep['geometry']
}, crs=padep.crs).to_crs('EPSG:26917')

print(f"PADEP wells: {len(padep_slim)}")

# --- Source 2: USGS National Orphaned Wells CSV ---
csv_path = r'C:\sp\US_orphaned_wells.csv'
print(f"\nLoading CSV: {csv_path}")

usgs = pd.read_csv(csv_path, low_memory=False)

# Filter to orphaned/abandoned status only
target_statuses = [
    'Orphan', 'orphan', 'Orphaned',
    'OR',
    'Abandoned', 'Abandoned Well', 'Abandoned Orphaned Well',
    'ACT 404 ORPHAN WELL-ENG',
    'ACT 404 ORPHAN WELL-INJECTION AND MINING'
]
usgs = usgs[usgs['Status'].isin(target_statuses)].copy()

# Drop rows with missing coordinates
usgs = usgs.dropna(subset=['Latitude', 'Longitude'])

# Convert to GeoDataFrame
usgs_gdf = gpd.GeoDataFrame(
    {
        'source': 'USGS_national',
        'state':  usgs['State'],
        'type':   usgs['Type'],
        'geometry': [Point(xy) for xy in zip(usgs['Longitude'], usgs['Latitude'])]
    },
    crs='EPSG:4326'
).to_crs('EPSG:26917')

print(f"USGS orphaned wells (filtered): {len(usgs_gdf)}")
print(f"States represented: {usgs_gdf['state'].nunique()}")

# --- Combine ---
wells = pd.concat([padep_slim, usgs_gdf], ignore_index=True)
wells = gpd.GeoDataFrame(wells, crs='EPSG:26917')

print(f"\nCombined total: {len(wells)}")
print(f"\nBy source:")
print(wells['source'].value_counts())

Loading shapefile: C:\Users\colto\Documents\PADEP_HistoricOilGasWells_ALL\PADEP_HistoricOilGasWells_ALL.shp
PADEP wells: 30527

Loading CSV: C:\sp\US_orphaned_wells.csv
USGS orphaned wells (filtered): 79456
States represented: 16

Combined total: 109983

By source:
source
USGS_national     79456
PADEP_historic    30527
Name: count, dtype: int64


In [16]:
import rasterio
from shapely.geometry import box

# What well types do we have?
print("Well types:")
print(wells['type'].value_counts().head(20))

# Already in EPSG:26917 from the previous cell
wells_utm = wells.copy()
print(f"\nCRS: {wells_utm.crs}")

# Get DEM extent and clip wells to it
with rasterio.open(FULL_DEM) as src:
    dem_bounds = src.bounds
    dem_crs = src.crs

dem_bbox = box(dem_bounds.left, dem_bounds.bottom, dem_bounds.right, dem_bounds.top)
dem_gdf = gpd.GeoDataFrame(geometry=[dem_bbox], crs=dem_crs)

# Spatial clip — only wells inside LiDAR coverage
wells_clipped = gpd.clip(wells_utm, dem_gdf)
print(f"\nWells in full dataset:      {len(wells_utm)}")
print(f"Wells inside LiDAR extent:  {len(wells_clipped)}")
print(f"\nBy source in clipped area:")
print

Well types:
type
Oil Well                         13180
Gas Well                          9796
OIL                               8726
O                                 5365
GAS                               4890
Dry Hole                          3964
Well, No Record                   3189
WI                                2448
NT                                2444
Vertical                          2140
Not Available                     1454
DRY                               1294
Abandoned Gas Well                1147
Gas(Convertional, Commercial)     1055
Stratigraphic Test                1042
NO PRODUCT SPECIFIED              1019
OG                                1004
TM                                 958
CBM Well                           934
Oil                                707
Name: count, dtype: int64

CRS: EPSG:26917

Wells in full dataset:      109983
Wells inside LiDAR extent:  3426

By source in clipped area:


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

EXTRACT FEATURES FROM WELL


In [17]:
# from rasterstats import point_query
# import pandas as pd
#
# wells_clipped = wells_clipped.reset_index(drop=True)
#
# print("Extracting features at well locations...")
# well_features = {}
# for name, path in rasters.items():
#     well_features[name] = point_query(wells_clipped, path)
#
# well_df = pd.DataFrame(well_features)
# well_df['label'] = 1
# well_df['type'] = wells_clipped['type'].values
#
# print(f"Well samples: {len(well_df)}")
# print(f"NaN count per feature:")
# print(well_df.drop(columns=['label','type']).isna().sum())
# print(well_df.head())

In [18]:
from rasterstats import point_query
import pandas as pd

# All derivative rasters
rasters = {
    'tpi_5m':    os.path.join(DERIV_FOLDER, 'tpi_5m.tif'),
    'tpi_15m':   os.path.join(DERIV_FOLDER, 'tpi_15m.tif'),
    'tpi_50m':   os.path.join(DERIV_FOLDER, 'tpi_50m.tif'),
    'slope':     os.path.join(DERIV_FOLDER, 'slope.tif'),
    'curvature': os.path.join(DERIV_FOLDER, 'curvature.tif'),
    'relief':    os.path.join(DERIV_FOLDER, 'relief_10m.tif'),
    'roughness': os.path.join(DERIV_FOLDER, 'roughness.tif'),
}

# Extract features at well locations
print("Extracting features at well locations...")
well_features = {}
for name, path in rasters.items():
    well_features[name] = point_query(wells_clipped, path)

well_df = pd.DataFrame(well_features)
well_df['label'] = 1
well_df['type'] = wells_clipped['type'].values

print(f"Well samples: {len(well_df)}")
print(f"NaN count per feature:")
print(well_df.drop(columns=['label','type']).isna().sum())
print(f"\nSample:")
print(well_df.head())

Extracting features at well locations...
Well samples: 3426
NaN count per feature:
tpi_5m       37
tpi_15m      37
tpi_50m      37
slope        37
curvature    37
relief       37
roughness    37
dtype: int64

Sample:
     tpi_5m   tpi_15m   tpi_50m     slope  curvature    relief  roughness  \
0       NaN       NaN       NaN       NaN        NaN       NaN        NaN   
1       NaN       NaN       NaN       NaN        NaN       NaN        NaN   
2 -0.058247 -0.140096  1.198271  5.889596   0.100206  3.386711   1.102475   
3  0.081193  0.184802  0.246708  2.991245  -0.130112  2.738085   1.398322   
4       NaN       NaN       NaN       NaN        NaN       NaN        NaN   

   label type  
0      1  NaN  
1      1  NaN  
2      1  NaN  
3      1  NaN  
4      1  NaN  


NEGATIVE SAMPLING AND FEATURE EXTRACTION

In [19]:
import numpy as np
from shapely.geometry import Point
import random

well_df_clean = well_df.dropna(subset=list(rasters.keys())).reset_index(drop=True)
print(f"Usable well samples: {len(well_df_clean)}")

with rasterio.open(FULL_DEM) as src:
    bounds   = src.bounds
    dem_arr  = src.read(1)
    nodata_v = src.nodata
    transform = src.transform

valid_mask_flat = dem_arr != nodata_v
all_well_buffer = wells_clipped.geometry.buffer(50).union_all()

random.seed(42)
non_well_points = []
attempts = 0
max_attempts = 200000

while len(non_well_points) < len(well_df_clean) and attempts < max_attempts:
    x  = random.uniform(bounds.left,   bounds.right)
    y  = random.uniform(bounds.bottom, bounds.top)
    pt = Point(x, y)
    if not all_well_buffer.contains(pt):
        col, row = ~transform * (x, y)
        col, row = int(col), int(row)
        if 0 <= row < dem_arr.shape[0] and 0 <= col < dem_arr.shape[1]:
            if valid_mask_flat[row, col]:
                non_well_points.append(pt)
    attempts += 1

print(f"Generated {len(non_well_points)} negative samples ({attempts} attempts)")

non_wells_gdf = gpd.GeoDataFrame(geometry=non_well_points, crs=wells_clipped.crs)

# Fix: convert to list of WKT strings for rasterstats compatibility
print("Extracting features at non-well locations...")
nonwell_features = {}
for name, path in rasters.items():
    nonwell_features[name] = point_query(non_wells_gdf, path)

nonwell_df = pd.DataFrame(nonwell_features)
nonwell_df['label'] = 0
nonwell_df['type']  = 'non_well'
nonwell_df = nonwell_df.dropna().reset_index(drop=True)

all_samples = pd.concat([well_df_clean, nonwell_df], ignore_index=True)
print(f"\nFinal dataset:")
print(f"  Wells (label=1):     {(all_samples['label']==1).sum()}")
print(f"  Non-wells (label=0): {(all_samples['label']==0).sum()}")
print(f"  Total:               {len(all_samples)}")

Usable well samples: 3389
Generated 3389 negative samples (3732 attempts)
Extracting features at non-well locations...

Final dataset:
  Wells (label=1):     3389
  Non-wells (label=0): 3389
  Total:               6778


In [20]:
# import numpy as np
# from shapely.geometry import Point
# import random
#
# # Drop NaN wells first
# well_df_clean = well_df.dropna().reset_index(drop=True)
# wells_clean = wells_clipped.dropna(subset=['geometry']).reset_index(drop=True)
# print(f"Usable well samples: {len(well_df_clean)}")
#
# # Get DEM bounds and valid mask for negative sampling
# with rasterio.open(FULL_DEM) as src:
#     bounds = src.bounds
#     dem_arr = src.read(1)
#     nodata_v = src.nodata
#     transform = src.transform
#
# valid_mask_flat = dem_arr != nodata_v
#
# # Buffer around ALL wells (including NaN ones) to exclude from negative sampling
# all_well_buffer = wells_clipped.geometry.buffer(50).unary_union
#
# # Generate random non-well points
# random.seed(42)
# non_well_points = []
# attempts = 0
# max_attempts = 50000
#
# while len(non_well_points) < len(well_df_clean) and attempts < max_attempts:
#     x = random.uniform(bounds.left, bounds.right)
#     y = random.uniform(bounds.bottom, bounds.top)
#     pt = Point(x, y)
#
#     # Must be outside well buffers
#     if not all_well_buffer.contains(pt):
#         # Must be inside valid DEM data
#         col, row = ~transform * (x, y)
#         col, row = int(col), int(row)
#         if 0 <= row < dem_arr.shape[0] and 0 <= col < dem_arr.shape[1]:
#             if valid_mask_flat[row, col]:
#                 non_well_points.append(pt)
#     attempts += 1
#
# print(f"Generated {len(non_well_points)} negative samples ({attempts} attempts)")
#
# non_wells_gdf = gpd.GeoDataFrame(geometry=non_well_points, crs=wells_clipped.crs)
#
# # Extract features at negative locations
# print("Extracting features at non-well locations...")
# nonwell_features = {}
# for name, path in rasters.items():
#     nonwell_features[name] = point_query(non_wells_gdf, path)
#
# nonwell_df = pd.DataFrame(nonwell_features)
# nonwell_df['label'] = 0
# nonwell_df['type'] = 'non_well'
# nonwell_df = nonwell_df.dropna().reset_index(drop=True)
#
# # Combine
# all_samples = pd.concat([well_df_clean, nonwell_df], ignore_index=True)
# print(f"\nFinal dataset:")
# print(f"  Wells (label=1):     {(all_samples['label']==1).sum()}")
# print(f"  Non-wells (label=0): {(all_samples['label']==0).sum()}")
# print(f"  Total:               {len(all_samples)}")

In [ ]:
from rasterstats import point_query, zonal_stats
import pandas as pd

def extract_patch_features(gdf, rasters, radius_m=15):
    """Extract mean and min over a circular buffer around each point."""
    buffered = gdf.copy()
    buffered['geometry'] = gdf.geometry.buffer(radius_m)

    all_features = {}
    for name, path in rasters.items():
        stats = zonal_stats(buffered, path, stats=['mean', 'min', 'std'], nodata=np.nan)
        all_features[f'{name}_mean'] = [s['mean'] for s in stats]
        all_features[f'{name}_min']  = [s['min']  for s in stats]
        all_features[f'{name}_std']  = [s['std']  for s in stats]

    return pd.DataFrame(all_features)

print("Extracting patch features at well locations...")
well_patch_df = extract_patch_features(wells_clipped, rasters, radius_m=15)
well_patch_df['label'] = 1
well_patch_df['type']  = wells_clipped['type'].values

print("Extracting patch features at non-well locations...")
nonwell_patch_df = extract_patch_features(non_wells_gdf, rasters, radius_m=15)
nonwell_patch_df['label'] = 0
nonwell_patch_df['type']  = 'non_well'

# Combine and clean
all_patch = pd.concat([well_patch_df, nonwell_patch_df], ignore_index=True)
all_patch = all_patch.dropna().reset_index(drop=True)

print(f"\nFinal patch dataset:")
print(f"  Wells:     {(all_patch['label']==1).sum()}")
print(f"  Non-wells: {(all_patch['label']==0).sum()}")
print(f"  Features:  {[c for c in all_patch.columns if c not in ['label','type']]}")

Extracting patch features at well locations...
Extracting patch features at non-well locations...


In [ ]:
# ── CLASSIFIER CONFIGURATION ─────────────────────────────────────────────────
# Change CLASSIFIER and re-run this cell + the next one to switch models.

CLASSIFIER  = 'random_forest'   # 'random_forest' | 'xgboost'
FEATURE_SET = 'point'           # 'point'  — 7 point-sampled features
                                # 'patch'  — 21 patch stats (requires all_patch)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import pandas as pd

# ── Feature preparation ───────────────────────────────────────────────────────
if FEATURE_SET == 'patch':
    dataset      = all_patch
    feature_cols = [c for c in all_patch.columns if c not in ['label', 'type']]
else:
    dataset      = all_samples
    feature_cols = ['tpi_5m', 'tpi_15m', 'tpi_50m', 'slope', 'curvature', 'relief', 'roughness']

X = dataset[feature_cols]
y = dataset['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Classifier:  {CLASSIFIER}")
print(f"Features:    {FEATURE_SET}  ({len(feature_cols)})")
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

# ── Build model ───────────────────────────────────────────────────────────────
def build_model(name):
    if name == 'random_forest':
        return RandomForestClassifier(
            n_estimators=200, random_state=42, n_jobs=-1
        )
    if name == 'xgboost':
        return XGBClassifier(
            n_estimators=200, learning_rate=0.1, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1
        )
    raise ValueError(f"Unknown classifier: {name!r}")

model = build_model(CLASSIFIER)
model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

print(f"\n--- {CLASSIFIER.upper()}  |  {FEATURE_SET} features ---")
print(classification_report(y_test, y_pred, target_names=['non-well', 'well']))

# ── Visualize ─────────────────────────────────────────────────────────────────
importances = model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['non-well', 'well'])
disp.plot(ax=axes[0], colorbar=False, cmap='cividis')
axes[0].set_title(f'Confusion Matrix — {CLASSIFIER}')

imp = pd.Series(importances, index=feature_cols).sort_values()
axes[1].barh(imp.index, imp.values, color='steelblue')
axes[1].set_xlabel('Importance')
axes[1].set_title('Feature Importance')
axes[1].axvline(1 / len(feature_cols), color='orange', linestyle='--', label='Uniform baseline')
axes[1].legend()

plt.tight_layout()
out_path = os.path.join(r'C:\users\colto\documents\github\lidar_project\data',
                        f'model_results_{CLASSIFIER}.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# Rebuild by joining on index properly
wells_export = wells_clipped.copy().reset_index(drop=True)

# Add feature columns — use NaN for the 58 dropped rows
for col in feature_cols:
    wells_export[col] = well_df[col].values  # use well_df (172 rows), not well_df_clean

wells_export.to_file(r'C:\users\colto\documents\github\lidar_project\data\wells_with_features.shp')
print("Exported.")
print(f"Total rows: {len(wells_export)}")
print(f"Rows with valid features: {wells_export['tpi_5m'].notna().sum()}")

In [ ]:
from scipy.ndimage import uniform_filter, maximum_filter, minimum_filter, generic_filter

# Use a crop that's fully inside valid data to avoid edge effects
crop      = dem_filled[500:1000, 500:1000]
mask_crop = valid_mask[500:1000, 500:1000]

results = {}

# ── TPI (identical method, just verify) ──────────────────────────────────────
for radius, name in [(5, 'tpi_5m'), (15, 'tpi_15m'), (50, 'tpi_50m')]:
    old = dem_filled[500:1000, 500:1000] - uniform_filter(crop, size=radius*2+1)
    new = old.copy()  # identical method — should be 0
    old[~mask_crop] = np.nan
    new[~mask_crop] = np.nan
    results[name] = np.nanmax(np.abs(old - new))

# ── Slope & Curvature (identical method, just verify) ────────────────────────
def slope_curv(dem, mask):
    d         = np.pad(dem, 1, mode='edge')
    dz_dx     = (d[1:-1, 2:]  - d[1:-1, :-2]) / 2
    dz_dy     = (d[2:,  1:-1] - d[:-2,  1:-1]) / 2
    slope     = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
    d2z_dx2   = d[1:-1, 2:] - 2*d[1:-1,1:-1] + d[1:-1,:-2]
    d2z_dy2   = d[2:,1:-1]  - 2*d[1:-1,1:-1] + d[:-2, 1:-1]
    curv      = d2z_dx2 + d2z_dy2
    slope[~mask] = np.nan
    curv[~mask]  = np.nan
    return slope, curv

slope_crop, curv_crop = slope_curv(crop, mask_crop)
results['slope']     = 0.0   # identical method
results['curvature'] = 0.0

# ── Relief ───────────────────────────────────────────────────────────────────
old_relief = generic_filter(crop, np.nanmax, size=21) - \
             generic_filter(crop, np.nanmin, size=21)
new_relief = maximum_filter(crop, size=21) - minimum_filter(crop, size=21)
old_relief[~mask_crop] = np.nan
new_relief[~mask_crop] = np.nan
results['relief'] = np.nanmax(np.abs(old_relief - new_relief))

# ── Roughness ────────────────────────────────────────────────────────────────
old_roughness = generic_filter(slope_crop, np.nanstd, size=11)

fill_val          = float(np.nanmean(slope_crop[mask_crop]))
slope_fill        = slope_crop.copy()
slope_fill[~mask_crop] = fill_val
new_roughness     = np.sqrt(np.maximum(
    uniform_filter(slope_fill**2, size=11) - uniform_filter(slope_fill, size=11)**2, 0
))
old_roughness[~mask_crop] = np.nan
new_roughness[~mask_crop] = np.nan
results['roughness'] = np.nanmax(np.abs(old_roughness - new_roughness))

# ── Report ───────────────────────────────────────────────────────────────────
print(f"{'Derivative':<12}  {'Max Abs Diff':>14}  {'Verdict'}")
print("-" * 42)
for name, diff in results.items():
    verdict = "IDENTICAL" if diff < 1e-4 else "CLOSE (boundary effect)" if diff < 1.0 else "INVESTIGATE"
    print(f"{name:<12}  {diff:>14.6f}  {verdict}")